In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

from careamics.careamist import CAREamist
from careamics.config.factories import create_advanced_seg_config

In [ ]:
exp = "multi" # single, multi
loss = "dice_ce" # dice, dice_ce, ce

# ---
exp_name = f"seg_{exp}_{loss}"
root = Path("data")
if exp == "single":
    path = root / "single"
    n_classes = 1
else:
    path = root / "multi"
    n_classes = 2

x_train = np.load(path / "train.npy")
y_train = np.load(path / "train_target.npy")

x_val = np.load(path / "val.npy")
y_val = np.load(path / "val_target.npy")

test_files = sorted((path / "test").glob("*.npy"))
x_test = [np.load(t) for t in test_files]
y_test = [np.load(path / "test_target" / t.name) for t in test_files]

In [ ]:
# create a configuration
config = create_advanced_seg_config(
    experiment_name=exp_name,
    data_type="array",
    axes="SYX",
    patch_size=[128, 128],
    batch_size=16,
    n_channels_in=1,
    n_classes=n_classes,
    num_epochs=10,
    num_steps=500,
    normalization_params={
        "skip_target": True
    }
)

# instantiate a careamist
careamist = CAREamist(config, work_dir=root / "seg_results")
careamist.train(
    train_data=x_train,
    train_data_target=y_train,
)

In [ ]:
# -- plot training/val loss

losses = careamist.get_losses()

fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))

ax[0].plot(losses.train_loss.epoch, losses.train_loss.value, label="Train")
ax[0].plot(losses.val_loss.epoch, losses.val_loss.value, label="Validation")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].set_title("Losses")
ax[0].legend(frameon=False)

for k in losses.metrics.keys():
    metrics = losses.metrics[k]
    ax[1].plot(metrics.epoch, metrics.value, label=k)
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Dice score")
ax[1].set_title("Metrics")
ax[1].legend(frameon=False)

fig.tight_layout()

In [ ]:
# predict on test
preds, _ = careamist.predict(
    pred_data=x_test,
    axes="YX",
    target_axes="YX",
    batch_size=1
)


In [ ]:
# -- metrics
def compute_iou(pred: np.ndarray, tgt: np.ndarray, include_background: bool = True):
    """
    Compute per-class IoU and mean IoU from class-label predictions and targets.

    Returns
    -------
    tuple[np.ndarray, float]
        Per-class IoU and mean IoU.
    """
    num_classes = int(max(pred.max(), tgt.max())) + 1
    iou_per_class = np.zeros(num_classes, dtype=np.float64)

    for c in range(num_classes):
        pred_mask = pred == c
        tgt_mask = tgt == c

        intersection = np.logical_and(pred_mask, tgt_mask).sum()
        union = np.logical_or(pred_mask, tgt_mask).sum()

        iou_per_class[c] = intersection / union if union > 0 else np.nan

    if include_background:
        mean_iou = np.nanmean(iou_per_class)
    else:
        mean_iou = np.nanmean(iou_per_class[1:])

    return iou_per_class, mean_iou

iou = {}
for i, p in enumerate(preds):
    iou_per_class, mean_iou = compute_iou(p, y_test[i])

    for c in range(n_classes+1):

        if c in iou.keys():
            iou[c].append(iou_per_class[c])
        else:
            iou[c] = [iou_per_class[c]]

strs = ["Results (IoU)\n"]
for k in iou:
    strs.append(f"Class {k}: {np.mean(iou[k])} +/- {np.std(iou[k])}\n")

path = root / "seg_results" / exp_name
path.mkdir(parents=True, exist_ok=True)
with open(path / "results.txt", mode="w") as f:
    f.writelines(strs)

for line in strs:
    print(line[:-1])


In [ ]:
# -- plot results
sample = 5
n_pred = len(preds)
rng = np.random.default_rng()
idx = rng.choice(n_pred, size=sample)

fig, axes = plt.subplots(sample, 3, figsize=(10, 15))
for i, k in enumerate(idx):
    _, mean_iou = compute_iou(preds[k], y_test[k])


    axes[i, 0].imshow(x_test[k], cmap="gray")
    axes[i, 0].set_title("Raw")
    axes[i, 1].imshow(preds[k])
    axes[i, 1].set_title(f"Prediction {k}\nmIoU {mean_iou:.2f}")
    axes[i, 2].imshow(y_test[k])
    axes[i, 2].set_title("Ground Truth")
plt.tight_layout()
plt.show()


In [ ]:
# --- save all predictions

path = root / "seg_results" / exp_name / "predictions"
path.mkdir(exist_ok=True, parents=True)
for i, p in enumerate(preds):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(x_test[i], cmap="gray")
    axes[0].set_title("Raw")
    axes[1].imshow(p)
    axes[1].set_title("Prediction")
    axes[2].imshow(y_test[i])
    axes[2].set_title("Ground Truth")
    plt.savefig(path / f"sample_{i}.png")
    plt.close(fig)